In [ ]:
# Hyperparameter tuning: feature engineering (domain knowledge feature) ---> encoding ---> cross validation ---> change paramters ---> re-evaluate models 
# Imports
import os
import sys
import numpy as np
from huggingface_hub import login
# Add the utils folder to Python path
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
pricer_path = os.path.join(notebook_dir, "../utils")
sys.path.insert(0, pricer_path)
# Class Item to validate the returned data and has lots of helper functions
from pricer.items import Item
from pricer.evaluator import evaluate
# Machine learning models
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb 

In [2]:
# Login to Hugging Face
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# Download the batches dataset from Hugging Face 
username = "omarshaarawy11"
dataset = f"{username}/best_deal_03_batches"
train, val, test = Item.from_hub(dataset)
items = train + val + test
print(f"Loaded {len(items):,} items")

Loaded 21,091 items


In [4]:
# Invistigate the dataset
columns = ['title', 
'category', 
'price',
'full',
'weight',
'summary',
'prompt',
'id'
]
for i in columns:
    # Combine item with its attrs
    attr = getattr(items[0], i)
    print(attr)

Lurrose 100Pcs Full Cover Fake Toenails Artificial Transparent Nail Tips Nail Art for DIY
all beauty
6.99
None
0.0
Title: Lurrose 100pcs Full Cover Fake Toenails  
Category: Nail Art & Accessories  
Brand: Lurrose  
Description: High-quality, durable artificial toenails perfect for DIY nail art and manicures.  
Details: Features 100 lightweight, eco-friendly ABS nails that are easy to trim, shape, and fit comfortably to natural toenails, suitable for various styles and perfect as a gift.
None
None


In [5]:
# Feature engineering
# Domain knowledge feature
# Our candidate models for hyperparamter tuning are random forst and XGBoost
prices_train = np.array([float(item.price) for item in train])
prices_test = np.array([float(item.price) for item in test])
# Apply NLP 
# Add corpus
# Then we come up with the most k-words in corpus and treeat them as seperated features
documents_train = [item.summary for item in train]
documents_test = [item.summary for item in test]

In [6]:
# Encoding
# Apply count vectorizer to form new X_train and X_test
np.random.seed(40)
# Ignore stop English words, We need the important ones
vectorizer = CountVectorizer(max_features=450, stop_words='english')
X_train = vectorizer.fit_transform(documents_train)
X_test = vectorizer.transform(documents_test)

In [7]:
selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words 'famous' in both train and test: {len(selected_words)}")
print("Selected words in train and test:", selected_words)

Number of selected words 'famous' in both train and test: 450
Selected words in train and test: ['10' '100' '11' '12' '13' '14' '15' '16' '20' '24' '25' '30' '360' '4k'
 '50' 'ac' 'access' 'accessories' 'adapter' 'adhesive' 'adjustable' 'air'
 'airpods' 'aluminum' 'anti' 'apple' 'appliance' 'appliances'
 'approximately' 'art' 'assembly' 'audio' 'auto' 'available' 'backdrop'
 'backed' 'bag' 'bags' 'basket' 'bathroom' 'batteries' 'battery' 'bedding'
 'belt' 'black' 'blue' 'bluetooth' 'board' 'body' 'bpa' 'brand' 'brands'
 'breathable' 'built' 'burner' 'cable' 'cables' 'camera' 'cameras'
 'canvas' 'capacity' 'car' 'carbon' 'card' 'care' 'case' 'category'
 'ceramic' 'certified' 'charger' 'charging' 'christmas' 'clean' 'cleaning'
 'clear' 'closure' 'coffee' 'color' 'colorful' 'colors' 'comes' 'comfort'
 'comfortable' 'compact' 'compatibility' 'compatible' 'components'
 'comprehensive' 'connectivity' 'connector' 'connectors' 'construction'
 'control' 'controls' 'convenient' 'cooling' 'copper

In [8]:
X_train

<16800x450 sparse matrix of type '<class 'numpy.int64'>'
	with 367666 stored elements in Compressed Sparse Row format>

In [9]:
X_test

<2191x450 sparse matrix of type '<class 'numpy.int64'>'
	with 42425 stored elements in Compressed Sparse Row format>

In [10]:
# Modeling
# Hyperparamters tuning random forest
# Performance is worse than non-tuned version
np.random.seed(40)
y_train = prices_train
y_test = prices_test

# Train
model = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=4),
    param_distributions={
        "n_estimators": [100, 200],
        "max_depth": [None, 10, 20, 30, 40],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", None],
    },
    n_iter=5,
    scoring="neg_root_mean_squared_error",
    cv=2,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
model.fit(X_train, y_train)

# Predict 
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Evaluate 
# The best paramters
best_params = model.best_estimator_
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Best Parameters: {best_params}")
print(f"Train MSE: {train_mse: 0.2f}")
print(f"Train R2 Score: {train_r2: 0.2f}%")
print(f"Test MSE: {test_mse: 0.2f}")
print(f"Test R2 Score: {test_r2: 0.2f}%")

Fitting 2 folds for each of 5 candidates, totalling 10 fits
Best Parameters: RandomForestRegressor(max_depth=20, max_features='sqrt', min_samples_leaf=2,
                      min_samples_split=5, n_jobs=4, random_state=42)
Train MSE:  4535.18
Train R2 Score:  0.58%
Test MSE:  4563.19
Test R2 Score:  0.26%


In [11]:
# Test dataset
def random_forest_hyper_tuning(item):
    X_test = vectorizer.transform([item.summary])
    # In case we got - result then it gonna be zero
    return max(model.predict(X_test)[0], 0)  

In [12]:
# Test single datapoint
random_forest_hyper_tuning(items[0])

26.09393089746106

In [13]:
# Evaluation
evaluate(random_forest_hyper_tuning, test)

  0%|          | 0/2000 [00:00<?, ?it/s]

$18 $85 $25 $10 $39 $36 $13 $16 $9 $11 $1 $29 $27 $5 $29 $8 $18 $5 $302 $28 $16 $23 $49 $19 $21 $23 $14 $1 $12 $12 $14 $25 $25 $736 $7 $58 $28 $3 $17 $10 $39 $135 $23 $31 $33 $44 $24 $29 $14 $8 $44 $47 $27 $17 $17 $26 $22 $42 $29 $24 $77 $20 $28 $18 $95 $100 $30 $16 $17 $16 $26 $14 $14 $22 $7 $10 $74 $23 $28 $206 $44 $9 $20 $3 $30 $48 $2 $39 $4 $24 $33 $17 $27 $28 $2 $25 $16 $17 $44 $21 $15 $52 $26 $31 $28 $26 $18 $29 $12 $14 $37 $9 $64 $28 $7 $47 $13 $84 $20 $5 $32 $0 $32 $10 $19 $14 $3 $23 $22 $47 $6 $15 $177 $91 $12 $13 $14 $24 $10 $13 $14 $36 $15 $26 $21 $20 $20 $16 $22 $11 $34 $18 $6 $19 $42 $4 $37 $16 $94 $11 $7 $26 $17 $23 $42 $231 $23 $34 $44 $25 $19 $29 $115 $130 $14 $80 $58 $25 $40 $7 $29 $19 $26 $2 $157 $3 $21 $42 $29 $12 $5 $10 $16 $18 $31 $21 $28 $12 $27 $3 $33 $11 $9 $57 $24 $22 $57 $12 $1 $12 $15 $25 $13 $17 $13 $138 $23 $11 $7 $19 $30 $15 $23 $30 $7 $49 $45 $8 $4 $15 $30 $22 $9 $25 $39 $31 $77 $25 $14 $14 $2 $23 $19 $25 $20 $24 $28 $29 $53 $338 $44 $24 $1 $17 $67 $520 $

In [ ]:
# Modeling
# Hyperparamters tuning XGBoost
# Performance is worse than non-tuned version
np.random.seed(40)
y_train = prices_train
y_test = prices_test

# Train
model = RandomizedSearchCV(
    estimator=xgb.XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=4,
        eval_metric="rmse"
    ),
    param_distributions={
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 7, 9],
        "learning_rate": [0.01, 0.05, 0.1, 0.2],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "gamma": [0, 0.1, 0.5, 1],
        "min_child_weight": [1, 3, 5, 7],
    },
    n_iter=5,
    scoring="neg_root_mean_squared_error",
    cv=2,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
model.fit(X_train, y_train)

# Predict 
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Evaluate 
# The best paramters
best_params = model.best_estimator_
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Best Parameters: {best_params}")
print(f"Train MSE: {train_mse: 0.2f}")
print(f"Train R2 Score: {train_r2: 0.2f}%")
print(f"Test MSE: {test_mse: 0.2f}")
print(f"Test R2 Score: {test_r2: 0.2f}%")

Fitting 2 folds for each of 5 candidates, totalling 10 fits
Best Parameters: XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=1.0, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=0.5, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.2, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=5, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=4,
             num_parallel_tree=None, random_state=42, ...)
Train MSE:  3258.58
Train R2 Score:  0.70%
Test MSE:  4295.90
Test R2 Score:  0.31%


In [15]:
# Test dataset
def xgb_hyper_tuning(item):
    X_test = vectorizer.transform([item.summary])
    # In case we got - result then it gonna be zero
    return max(model.predict(X_test)[0], 0)  

In [16]:
# Test single datapoint
xgb_hyper_tuning(items[0])

7.699655

In [17]:
# Evaluation
evaluate(xgb_hyper_tuning, test)

  0%|          | 0/2000 [00:00<?, ?it/s]

$34 $90 $11 $1 $22 $39 $2 $17 $3 $4 $12 $3 $33 $8 $29 $1 $5 $8 $316 $114 $45 $27 $24 $16 $1 $9 $6 $12 $6 $8 $5 $14 $9 $731 $10 $49 $23 $10 $46 $11 $117 $165 $7 $33 $21 $27 $30 $12 $2 $14 $40 $31 $26 $1 $11 $20 $21 $2 $15 $22 $54 $21 $27 $18 $46 $106 $15 $12 $12 $18 $31 $1 $11 $2 $9 $1 $84 $20 $36 $206 $28 $8 $10 $96 $34 $32 $2 $37 $10 $12 $14 $9 $15 $35 $14 $12 $8 $16 $28 $3 $8 $79 $22 $23 $32 $64 $9 $7 $20 $11 $91 $16 $64 $70 $14 $57 $6 $90 $6 $24 $31 $5 $24 $2 $58 $27 $7 $19 $11 $32 $7 $4 $97 $61 $0 $4 $5 $32 $2 $2 $5 $1 $0 $19 $32 $0 $9 $22 $11 $27 $14 $18 $14 $8 $5 $6 $28 $14 $55 $10 $19 $39 $2 $14 $60 $238 $23 $58 $51 $21 $2 $23 $2 $157 $13 $119 $49 $8 $34 $1 $26 $23 $18 $8 $107 $0 $46 $39 $19 $0 $29 $15 $36 $6 $27 $20 $1 $1 $17 $12 $30 $2 $15 $42 $3 $27 $39 $1 $7 $3 $2 $53 $34 $10 $4 $140 $26 $37 $7 $15 $34 $3 $17 $21 $9 $51 $35 $17 $10 $8 $39 $21 $5 $15 $28 $24 $135 $12 $5 $3 $41 $5 $15 $3 $12 $3 $19 $0 $75 $261 $48 $25 $1 $13 $30 $474 $125 $3 $16 $20 $8 $34 $23 $15 $18 $11 $2 $